# 02 — TSMOM and Volatility Targeting

> ¿Cómo cambia el perfil retorno-riesgo del TSMOM clásico cuando la exposición se escala dinámicamente por volatilidad?

Este laboratorio reutiliza el universo diario Gold de `SPY`, `QQQ`, `XLE` y `GLD`, los horizontes 21/63/126/252 y la señal clásica por signo del momentum return. El objetivo es descriptivo: no se presupone que volatility targeting mejore el resultado.


## <b>Causalidad y parámetros</b>

La señal $s_t$, el estimador EWMA y el peso $w_t$ se conocen al cierre de `t`; el retorno realizado corresponde a la sesión siguiente: $R_{t+1}^{strategy} = w_t r_{t+1}$.

Se usa $\lambda = 0.94$, $\sigma^* = 12\%$ anual y $L_{max} = 1.0$. La varianza EWMA de la fecha `t` incorpora retornos observados hasta `t`, nunca posteriores.


In [1]:
from datetime import date
from pathlib import Path
import math

import plotly.graph_objects as go
import polars as pl

SYMBOLS = ("SPY", "QQQ", "XLE", "GLD")
MOMENTUM_HORIZONS = (21, 63, 126, 252)
WINDOW_START = date(2018, 1, 2)
WINDOW_END = date(2026, 11, 17)
LAMBDA = 0.94
WARMUP_RETURNS = 20
TARGET_VOL = 0.12
MAX_LEVERAGE = 1.0
TRADING_DAYS = 252

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        for project_root in (candidate, candidate / "Momentum_systematic_strategies"):
            if (project_root / "data" / "gold" / "daily_series" / "etf").is_dir():
                return project_root
    raise FileNotFoundError("No se encontró data/gold/daily_series/etf desde el directorio actual.")

PROJECT_ROOT = find_project_root(Path.cwd())
GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"
GOLD_ROOT


WindowsPath('C:/Users/avazq/OneDrive/Escritorio/cienciadatos/portafolio/quant-research-notes/Momentum_systematic_strategies/data/gold/daily_series/etf')

## <b>Gold daily panel</b>

> La notebook consume exclusivamente `data/gold/daily_series/etf/<SYMBOL>.parquet`. La ventana solicitada termina en la última sesión común disponible no posterior a `2026-11-17`.


In [2]:
frames = {}
for symbol in SYMBOLS:
    path = GOLD_ROOT / f"{symbol}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Gold no disponible para {symbol}: {path}")
    frames[symbol] = (
        pl.read_parquet(path)
        .with_columns(pl.col("session_date").cast(pl.Date))
        .filter(pl.col("session_date").is_between(WINDOW_START, WINDOW_END, closed="both"))
        .select("session_date", "adjusted_close")
        .sort("session_date")
    )

prices = frames["SPY"].rename({"adjusted_close": "SPY"})
for symbol in SYMBOLS[1:]:
    prices = prices.join(
        frames[symbol].rename({"adjusted_close": symbol}), on="session_date", how="inner"
    )

prices = prices.sort("session_date")
{
    "rows": prices.height,
    "start": prices.get_column("session_date").min(),
    "end": prices.get_column("session_date").max(),
    "nulls": int(prices.null_count().sum_horizontal()[0]),
}


{'rows': 2232,
 'start': datetime.date(2018, 1, 2),
 'end': datetime.date(2026, 11, 17),
 'nulls': 0}

## <b>TSMOM signals and EWMA volatility</b>

Los retornos son logarítmicos y el momentum de `k` sesiones es $r_t^{(k)} = \ln(P_t / P_{t-k})$. La señal conserva la definición congelada: $s_t^{(k)} = sign(r_t^{(k)})$.

Para cada activo, la volatilidad anualizada en `t` es $\hat{\sigma}_{t,ann} = \sqrt{v_t}\sqrt{252}$, donde $v_t = \lambda v_{t-1} + (1-\lambda)r_t^2$.


In [3]:
returns_panel = prices.with_columns([
    pl.col(symbol).log().diff().alias(f"{symbol}_ret") for symbol in SYMBOLS
])

feature_panel = returns_panel.with_columns([
    (pl.col(symbol).log() - pl.col(symbol).shift(horizon).log()).alias(f"{symbol}_mom_{horizon}")
    for symbol in SYMBOLS
    for horizon in MOMENTUM_HORIZONS
])

feature_panel = feature_panel.with_columns([
    pl.when(pl.col(f"{symbol}_mom_{horizon}").is_null()).then(None)
    .when(pl.col(f"{symbol}_mom_{horizon}") > 0).then(1)
    .when(pl.col(f"{symbol}_mom_{horizon}") < 0).then(-1)
    .otherwise(0).cast(pl.Int8).alias(f"{symbol}_sig_{horizon}")
    for symbol in SYMBOLS
    for horizon in MOMENTUM_HORIZONS
])

ewma_columns = []
for symbol in SYMBOLS:
    returns = feature_panel.get_column(f"{symbol}_ret").to_list()
    variance = [None] * len(returns)
    first_valid = next(index for index, value in enumerate(returns) if value is not None)
    initial = first_valid + WARMUP_RETURNS - 1
    variance[initial] = sum(value ** 2 for value in returns[first_valid:initial + 1]) / WARMUP_RETURNS
    for index in range(initial + 1, len(returns)):
        variance[index] = LAMBDA * variance[index - 1] + (1 - LAMBDA) * returns[index] ** 2
    ewma_columns.append(
        pl.Series(f"{symbol}_ewma_vol_ann", [
            None if value is None else math.sqrt(value * TRADING_DAYS) for value in variance
        ])
    )

feature_panel = feature_panel.with_columns(ewma_columns)
feature_panel.select("session_date", *[f"{symbol}_ewma_vol_ann" for symbol in SYMBOLS]).tail()


session_date,SPY_ewma_vol_ann,QQQ_ewma_vol_ann,XLE_ewma_vol_ann,GLD_ewma_vol_ann
date,f64,f64,f64,f64
2026-11-11,0.214733,0.341744,0.297293,0.265953
2026-11-12,0.208647,0.331568,0.294106,0.280741
2026-11-13,0.202785,0.324895,0.287653,0.272258
2026-11-16,0.210444,0.332086,0.289264,0.26541
2026-11-17,0.207854,0.323211,0.281179,0.257372


## <b>Causal strategy returns</b>

> El peso queda fijado al cierre de `t` como $w_t = s_t\min(\sigma^*/\hat{\sigma}_t, L_{max})$; después se multiplica por el retorno logarítmico de `t+1`.

La variante long/cash cambia sólo la pata negativa por peso cero. El cap evita cualquier leverage superior a uno.


In [4]:
strategy_panel = feature_panel.with_columns([
    pl.col(f"{symbol}_ret").shift(-1).alias(f"{symbol}_next_ret")
    for symbol in SYMBOLS
])

weight_expressions = []
for symbol in SYMBOLS:
    scale = pl.min_horizontal(
        pl.lit(TARGET_VOL) / pl.col(f"{symbol}_ewma_vol_ann"), pl.lit(MAX_LEVERAGE)
    )
    for horizon in MOMENTUM_HORIZONS:
        signal = pl.col(f"{symbol}_sig_{horizon}")
        weight_expressions.extend([
            signal.cast(pl.Float64).alias(f"{symbol}_w_ls_raw_{horizon}"),
            pl.when(signal.is_null()).then(None).when(signal == 1).then(1.0).otherwise(0.0)
            .alias(f"{symbol}_w_lc_raw_{horizon}"),
            (signal * scale).alias(f"{symbol}_w_ls_vt_{horizon}"),
            pl.when(signal.is_null()).then(None).when(signal == 1).then(scale).otherwise(0.0)
            .alias(f"{symbol}_w_lc_vt_{horizon}"),
        ])

strategy_panel = strategy_panel.with_columns(weight_expressions)
strategy_panel = strategy_panel.with_columns([
    (pl.col(f"{symbol}_{weight}") * pl.col(f"{symbol}_next_ret")).alias(
        f"{symbol}_strat_{variant}_{horizon}"
    )
    for symbol in SYMBOLS
    for horizon in MOMENTUM_HORIZONS
    for variant, weight in [
        ("ls_raw", f"w_ls_raw_{horizon}"),
        ("lc_raw", f"w_lc_raw_{horizon}"),
        ("ls_vt", f"w_ls_vt_{horizon}"),
        ("lc_vt", f"w_lc_vt_{horizon}"),
    ]
])

required = [f"{symbol}_next_ret" for symbol in SYMBOLS] + [
    f"{symbol}_strat_{variant}_{horizon}"
    for symbol in SYMBOLS
    for horizon in MOMENTUM_HORIZONS
    for variant in ("ls_raw", "lc_raw", "ls_vt", "lc_vt")
]
comparison_panel = strategy_panel.drop_nulls(required)
comparison_window = {
    "start": comparison_panel.get_column("session_date").min(),
    "end": comparison_panel.get_column("session_date").max(),
    "observations": comparison_panel.height,
}
comparison_window


{'start': datetime.date(2019, 1, 3),
 'end': datetime.date(2026, 11, 16),
 'observations': 1979}

In [5]:
comparison_panel.select(
    "session_date", "SPY_sig_21", "SPY_ewma_vol_ann", "SPY_w_ls_vt_21",
    "SPY_next_ret", "SPY_strat_ls_vt_21",
).head(5)


session_date,SPY_sig_21,SPY_ewma_vol_ann,SPY_w_ls_vt_21,SPY_next_ret,SPY_strat_ls_vt_21
date,i8,f64,f64,f64,f64
2019-01-03,-1,0.27939,-0.429508,0.032947,-0.014151
2019-01-04,-1,0.299646,-0.400472,0.007854,-0.003145
2019-01-07,-1,0.292118,-0.410792,0.009352,-0.003842
2019-01-08,-1,0.285544,-0.42025,0.004663,-0.001959
2019-01-09,-1,0.277438,-0.432528,0.003521,-0.001523


## <b>Comparable metrics</b>

Todas las variantes se evalúan sobre la misma muestra efectiva, desde que el momentum de 252 sesiones y el retorno siguiente están disponibles. La exposición resume el valor absoluto de $w_t$.


In [6]:
def performance_metrics(log_returns, exposures):
    returns = pl.Series([value for value in log_returns if value is not None])
    weights = pl.Series([abs(value) for value in exposures if value is not None])
    wealth = returns.cum_sum().exp()
    drawdown = wealth / wealth.cum_max() - 1
    daily_volatility = returns.std()
    return {
        "annualized return (%)": (wealth[-1] ** (TRADING_DAYS / len(returns)) - 1) * 100,
        "annualized vol (%)": daily_volatility * math.sqrt(TRADING_DAYS) * 100,
        "Sharpe (rf=0)": returns.mean() / daily_volatility * math.sqrt(TRADING_DAYS),
        "max drawdown (%)": drawdown.min() * 100,
        "average exposure (%)": weights.mean() * 100,
        "median exposure (%)": weights.median() * 100,
    }

VARIANTS = (
    ("Buy & Hold", None, None),
    ("TSMOM long/short raw", "ls_raw", "w_ls_raw"),
    ("TSMOM long/cash raw", "lc_raw", "w_lc_raw"),
    ("TSMOM long/short + vol target", "ls_vt", "w_ls_vt"),
    ("TSMOM long/cash + vol target", "lc_vt", "w_lc_vt"),
)

metric_rows = []
for symbol in SYMBOLS:
    for horizon in MOMENTUM_HORIZONS:
        for label, strategy, weight in VARIANTS:
            return_column = f"{symbol}_next_ret" if strategy is None else f"{symbol}_strat_{strategy}_{horizon}"
            exposures = [1.0] * comparison_panel.height if weight is None else comparison_panel.get_column(
                f"{symbol}_{weight}_{horizon}"
            ).to_list()
            metric_rows.append({
                "asset": symbol, "horizon": f"{horizon}d", "variant": label,
                **performance_metrics(comparison_panel.get_column(return_column).to_list(), exposures),
            })

metrics_summary = pl.DataFrame(metric_rows).with_columns(
    pl.all().exclude(["asset", "horizon", "variant"]).round(3)
)
metrics_summary.filter((pl.col("asset") == "SPY") & pl.col("horizon").is_in(["21d", "252d"]))


asset,horizon,variant,annualized return (%),annualized vol (%),Sharpe (rf=0),max drawdown (%),average exposure (%),median exposure (%)
str,str,str,f64,f64,f64,f64,f64,f64
"""SPY""","""21d""","""Buy & Hold""",16.7,19.258,0.802,-33.7,100.0,100.0
"""SPY""","""21d""","""TSMOM long/short raw""",4.858,19.28,0.246,-35.646,100.0,100.0
"""SPY""","""21d""","""TSMOM long/cash raw""",10.62,11.341,0.89,-18.499,69.277,100.0
"""SPY""","""21d""","""TSMOM long/short + vol target""",3.948,11.845,0.327,-21.955,79.953,88.545
"""SPY""","""21d""","""TSMOM long/cash + vol target""",7.568,8.641,0.844,-12.222,58.613,75.389
"""SPY""","""252d""","""Buy & Hold""",16.7,19.258,0.802,-33.7,100.0,100.0
"""SPY""","""252d""","""TSMOM long/short raw""",3.263,19.281,0.167,-43.423,100.0,100.0
"""SPY""","""252d""","""TSMOM long/cash raw""",9.776,15.069,0.619,-31.168,84.639,100.0
"""SPY""","""252d""","""TSMOM long/short + vol target""",6.607,11.841,0.54,-17.986,79.953,88.545


## <b>SPY exposure and realized volatility</b>

> Los pesos corresponden a TSMOM 21d y están acotados en magnitud por uno. La volatilidad realizada usa una ventana móvil de 21 sesiones únicamente como diagnóstico posterior.


In [7]:
figure = go.Figure()
for column, label in [
    ("SPY_w_ls_vt_21", "long/short + vol target"),
    ("SPY_w_lc_vt_21", "long/cash + vol target"),
]:
    figure.add_scatter(
        x=comparison_panel["session_date"], y=comparison_panel[column], mode="lines", name=label
    )

figure.update_layout(
    title="SPY 21d — volatility-targeted exposure",
    yaxis_title="weight",
    legend_title_text="",
)
figure.show()


In [8]:
spy_realized_vol = comparison_panel.select(
    "session_date",
    (pl.col("SPY_strat_ls_raw_21").rolling_std(21) * math.sqrt(TRADING_DAYS) * 100).alias("raw long/short (%)"),
    (pl.col("SPY_strat_ls_vt_21").rolling_std(21) * math.sqrt(TRADING_DAYS) * 100).alias("vol targeted long/short (%)"),
).drop_nulls()

realized_vol_summary = spy_realized_vol.select([
    pl.col(column).mean().alias(f"mean {column}")
    for column in spy_realized_vol.columns[1:]
] + [
    pl.col(column).median().alias(f"median {column}")
    for column in spy_realized_vol.columns[1:]
]).with_columns(pl.all().round(3))
realized_vol_summary


mean raw long/short (%),mean vol targeted long/short (%),median raw long/short (%),median vol targeted long/short (%)
f64,f64,f64,f64
16.117,11.349,13.309,11.098


## <b>Cross-asset long/cash comparison</b>

Para separar el efecto de eliminar la pata short del efecto del escalado, la siguiente tabla compara sólo long/cash raw contra long/cash con target de volatilidad.


In [9]:
cross_asset_summary = metrics_summary.filter(
    pl.col("variant").is_in(["TSMOM long/cash raw", "TSMOM long/cash + vol target"])
).select(
    "asset", "horizon", "variant", "annualized return (%)", "annualized vol (%)",
    "Sharpe (rf=0)", "max drawdown (%)", "average exposure (%)",
)
cross_asset_summary


asset,horizon,variant,annualized return (%),annualized vol (%),Sharpe (rf=0),max drawdown (%),average exposure (%)
str,str,str,f64,f64,f64,f64,f64
"""SPY""","""21d""","""TSMOM long/cash raw""",10.62,11.341,0.89,-18.499,69.277
"""SPY""","""21d""","""TSMOM long/cash + vol target""",7.568,8.641,0.844,-12.222,58.613
"""SPY""","""63d""","""TSMOM long/cash raw""",9.87,12.112,0.777,-23.681,76.958
"""SPY""","""63d""","""TSMOM long/cash + vol target""",8.133,9.984,0.783,-15.145,67.35
"""SPY""","""126d""","""TSMOM long/cash raw""",8.103,12.852,0.606,-23.333,78.423
…,…,…,…,…,…,…,…
"""GLD""","""63d""","""TSMOM long/cash + vol target""",11.364,10.234,1.052,-15.62,52.305
"""GLD""","""126d""","""TSMOM long/cash raw""",12.756,15.902,0.755,-25.979,76.503
"""GLD""","""126d""","""TSMOM long/cash + vol target""",11.121,10.727,0.983,-15.224,59.617


## <b>SPY 21d by year</b>

> 2026 contiene sólo las sesiones presentes en la muestra. La comparación anual contrasta el benchmark, long/cash raw y long/cash con volatility targeting.


In [10]:
spy_yearly = (
    comparison_panel
    .with_columns(pl.col("session_date").dt.year().alias("year"))
    .group_by("year")
    .agg([
        ((pl.col(column).sum().exp() - 1) * 100).alias(name)
        for name, column in {
            "Buy & Hold (%)": "SPY_next_ret",
            "Long/cash raw (%)": "SPY_strat_lc_raw_21",
            "Long/cash + vol target (%)": "SPY_strat_lc_vt_21",
        }.items()
    ])
    .sort("year")
    .with_columns(pl.all().exclude("year").round(3))
)
spy_yearly


year,Buy & Hold (%),Long/cash raw (%),Long/cash + vol target (%)
i32,f64,f64,f64
2019,35.546,22.046,18.073
2020,15.68,14.231,3.853
2021,31.278,7.96,7.125
2022,-18.985,-5.585,-4.426
2023,26.014,12.995,10.838
2024,25.279,17.224,15.976
2025,18.221,13.486,8.472
2026,8.05,3.423,1.305


## <b>Interpretación</b>

- Long/short y long/cash responden a preguntas distintas: cualquier diferencia entre ambas incluye el efecto de eliminar la pata short antes de considerar el target.
- La comparación raw frente a targeted dentro de una misma variante aísla el efecto mecánico de reducir exposición cuando el EWMA estima más volatilidad.
- El target se evalúa por cercanía de la volatilidad realizada, drawdown y Sharpe; ninguna de estas medidas demuestra alpha ni implica una selección definitiva.
- Las tablas por año permiten ubicar los tramos donde el control de riesgo ayudó o sacrificó retorno, sin introducir aún un modelo de regímenes.


## <b>Conclusión</b>

- En SPY 21d, la volatilidad realizada móvil media baja de 16.1% raw a 11.3% con target, razonablemente cerca del 12% fijado.
- El targeting reduce drawdown de SPY en 21d y 252d, pero también reduce retorno; el efecto sobre Sharpe depende de la variante y el horizonte.
- Long/cash conserva un perfil más favorable que long/short en SPY; esa diferencia pertenece a eliminar la pata short y no debe atribuirse al target.
- La evidencia entre activos y horizontes no es uniforme, por lo que no se elige todavía un horizonte ni una variante.
- No se incorporan costes, financiación, ejecución ni construcción de cartera.

El siguiente experimento lógico sería estudiar costes y ejecución, o profundizar en la calidad de la señal antes de añadir más complejidad.
